# Classification Track (Part A) -- UrbanGlide Incident Type Classification

**Goal:** classify micro-mobility incidents (`Normal`, `Hard_Braking`, `Battery_Critical`, `Mechanical_Fault`, `Improper_Parking`) using the 5 Part-A algorithms required for Review 1. Part B (5 more algorithms) is added in Review 2.

**Owner:** Person B, both reviews.

---

### Section 0 -- Imports

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              confusion_matrix, ConfusionMatrixDisplay, classification_report,
                              roc_auc_score)

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 95
plt.rcParams['font.size'] = 10
sns.set_style('whitegrid')

print("Libraries loaded.")


Libraries loaded.


### Step 1.1 -- Dataset loading & audit

For a classification problem, the audit has to include the class distribution, not just feature dtypes -- it determines every metric choice for the rest of the notebook.

In [2]:
clf_file = "../data/urbanglide_classification_incident.csv"
df_clf = pd.read_csv(clf_file)

print("=" * 65)
print("  CLASSIFICATION DATASET AUDIT")
print("=" * 65)
print(f"Shape: {df_clf.shape[0]:,} rows x {df_clf.shape[1]} columns\n")
print("Column dtypes:")
for col, dt in df_clf.dtypes.items():
    print(f"  {col:<30} {dt}")

mv = df_clf.isnull().sum()
miss = pd.DataFrame({'Missing Count': mv, 'Missing %': (mv/len(df_clf)*100).round(2)})[mv > 0]
print("\nMissing value profile:")
print(miss.to_string() if len(miss) else "  None")

print("\nTarget ('incident_type') class counts:")
print(df_clf['incident_type'].value_counts().to_string())


  CLASSIFICATION DATASET AUDIT
Shape: 12,030 rows x 14 columns

Column dtypes:
  ride_id                        int64
  vehicle_type                   str
  distance_km                    float64
  duration_min                   float64
  avg_speed_kmph                 float64
  max_speed_kmph                 float64
  battery_start_percent          float64
  battery_end_percent            float64
  sudden_stop_count              int64
  terrain_elevation_gain_m       float64
  temperature_c                  float64
  rider_age_group                str
  gps_signal_quality             float64
  incident_type                  str

Missing value profile:
                    Missing Count  Missing %
rider_age_group               108       0.90
gps_signal_quality            107       0.89

Target ('incident_type') class counts:
incident_type
Normal              6243
Hard_Braking        2863
Battery_Critical    1257
Mechanical_Fault     963
Improper_Parking     704


#### Output analysis

12,030 rows, missing values isolated to `rider_age_group`, `gps_signal_quality` at under 1% each. The class counts already hint at imbalance -- quantified properly in the next cell.